In [1]:
# Text Classification with Logistic Regression
# Dataset : `imdb_labelled.txt` (sentiment labels: 0 = negative, 1 = positive)  
# Task : Binary text classification using Logistic Regression.  
# Split : 80% training – 20% validation.

# 1. Import libraries

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

In [2]:
# 2. Load the dataset

df = pd.read_csv('imdb_labelled.txt', sep='\t', header=None, names=['text', 'label'])
df = df.dropna(subset=['label'])          # drop rows where label is missing
df['label'] = df['label'].astype(int)     # ensure integer labels
print(f'Dataset size: {df.shape[0]}')

Dataset size: 748


In [3]:
# 3. Basic data inspection

print("Label distribution:")
print(df['label'].value_counts())
print(f"\nMissing values:\n{df.isnull().sum()}")


Label distribution:
label
1    386
0    362
Name: count, dtype: int64

Missing values:
text     0
label    0
dtype: int64


In [4]:
# 4. Split into training and validation sets (80/20)
# 
# We use stratified splitting to keep the same class proportion in both sets.

X = df['text']
y = df['label']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Training label distribution:\n{y_train.value_counts(normalize=True)}")
print(f"Validation label distribution:\n{y_val.value_counts(normalize=True)}")



Training samples: 598
Validation samples: 150
Training label distribution:
label
1    0.516722
0    0.483278
Name: proportion, dtype: float64
Validation label distribution:
label
1    0.513333
0    0.486667
Name: proportion, dtype: float64


In [5]:
# 5. Text vectorisation with TF‑IDF
# We fit the vectoriser **only on the training data** to avoid data leakage.

vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

print(f"TF‑IDF feature matrix shape (train): {X_train_tfidf.shape}")


TF‑IDF feature matrix shape (train): (598, 2411)


In [6]:
# 6. Train a Logistic Regression classifier

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_tfidf, y_train)

print("Model training completed.")


Model training completed.


In [7]:
# 7. Evaluate the model on the validation set

y_pred = model.predict(X_val_tfidf)

# Accuracy
accuracy = accuracy_score(y_val, y_pred)
print(f"Validation Accuracy: {accuracy:.4f}")

# Classification Report
print("\nClassification Report:")
print(classification_report(y_val, y_pred, target_names=['Negative', 'Positive']))

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred))


Validation Accuracy: 0.7667

Classification Report:
              precision    recall  f1-score   support

    Negative       0.77      0.74      0.76        73
    Positive       0.76      0.79      0.78        77

    accuracy                           0.77       150
   macro avg       0.77      0.77      0.77       150
weighted avg       0.77      0.77      0.77       150

Confusion Matrix:
[[54 19]
 [16 61]]
